# Despliegue con interfaz gráfica – Precio de Airbnb en Nueva York (2019)

Este notebook prepara el despliegue del modelo final del notebook *Modelo_Airbnb_NYC* en una aplicación web hecha con **Streamlit**, que se publica en **Streamlit Community Cloud** a partir de un repositorio de GitHub.

**Requisito:** el archivo `modelo_final_airbnb.pkl`, generado y descargado al final del notebook de modelado.

**Pasos**
1. Crear el archivo `features.py` (funciones de preparación de datos e ingeniería de características)
2. Cargar el modelo final
3. Probar el modelo con alojamientos de ejemplo
4. Crear la aplicación Streamlit (`app.py`)
5. Crear el archivo de requisitos (`requirements.txt`) y descargar los archivos para GitHub

# 1. Archivo `features.py`

El pipeline guardado usa las funciones `preparar_datos` y `crear_variables`. Para poder cargarlo, el archivo `features.py` debe estar en la misma carpeta que el modelo y ser **idéntico** al del notebook de modelado.

In [ ]:
%%writefile features.py
# Funciones de preparación de datos e ingeniería de características.
# Este archivo lo usan el notebook de modelado y la aplicación de Streamlit.
import numpy as np

# Variables que usa el modelo
COLUMNAS_MODELO = ['neighbourhood_group', 'latitude', 'longitude', 'room_type',
                   'minimum_nights', 'number_of_reviews', 'reviews_per_month',
                   'calculated_host_listings_count', 'availability_365']

# Variables de conteo con distribución muy sesgada
COLUMNAS_SESGADAS = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
                     'calculated_host_listings_count']


def distancia_centro_km(lat, lon):
    # Distancia en km hasta Times Square (fórmula de haversine)
    lat1, lon1 = np.radians(lat), np.radians(lon)
    lat2, lon2 = np.radians(40.7580), np.radians(-73.9855)
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def preparar_datos(df):
    # Mismas operaciones del notebook de preparación de datos
    df = df[COLUMNAS_MODELO].copy()                                # selección de variables
    df['reviews_per_month'] = df['reviews_per_month'].fillna(0)    # nulos: sin reseñas -> 0
    df['minimum_nights'] = df['minimum_nights'].clip(1, 365)       # regla de calidad
    df['availability_365'] = df['availability_365'].clip(0, 365)   # regla de calidad
    return df


def crear_variables(df):
    # Ingeniería de características
    df = df.copy()
    df['dist_centro_km'] = distancia_centro_km(df['latitude'], df['longitude'])
    for col in COLUMNAS_SESGADAS:
        df[col] = np.log1p(df[col])
    return df

# 2. Cargar el modelo final

Antes de ejecutar, subir el archivo `modelo_final_airbnb.pkl` al panel de archivos de Colab.

In [ ]:
import joblib
import pandas as pd

modelo = joblib.load('modelo_final_airbnb.pkl')
modelo

# 3. Prueba del modelo

El pipeline recibe los datos tal como los ingresaría un usuario: no hace falta calcular la distancia al centro, ni crear dummies, ni escalar. Todo eso ocurre dentro del pipeline.

In [ ]:
ejemplos = pd.DataFrame({
    'neighbourhood_group': ['Manhattan', 'Brooklyn', 'Bronx'],
    'latitude': [40.7549, 40.6782, 40.8448],
    'longitude': [-73.9840, -73.9442, -73.8648],
    'room_type': ['Entire home/apt', 'Private room', 'Shared room'],
    'minimum_nights': [2, 1, 1],
    'number_of_reviews': [25, 40, 5],
    'reviews_per_month': [1.2, 2.0, 0.3],
    'calculated_host_listings_count': [1, 1, 2],
    'availability_365': [150, 200, 300],
})

ejemplos['precio_estimado_USD'] = modelo.predict(ejemplos).round(0)
ejemplos[['neighbourhood_group', 'room_type', 'precio_estimado_USD']]

# 4. Aplicación Streamlit

La aplicación pide la ubicación **con un clic en un mapa interactivo** (librerías `folium` y `streamlit-folium`), el tipo de alojamiento y los datos del anfitrión, y muestra el precio estimado por noche. El modelo se carga una sola vez gracias a `@st.cache_resource`, lo que hace que la aplicación responda más rápido.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st
import folium
from streamlit_folium import st_folium

st.title('Predicción del precio de Airbnb en Nueva York')
st.write('Ingresa las características del alojamiento para estimar su precio por noche (datos de 2019).')


# Cargar el modelo una sola vez (el caché evita recargarlo en cada clic)
@st.cache_resource
def cargar_modelo():
    return joblib.load('modelo_final_airbnb.pkl')


modelo = cargar_modelo()

# ---------------- Ubicación: se elige con un clic en el mapa
st.header('Ubicación')
st.write('Haz clic en el mapa para marcar dónde está el alojamiento.')

# Ubicación inicial: Times Square
if 'ubicacion' not in st.session_state:
    st.session_state['ubicacion'] = [40.7580, -73.9855]

mapa = folium.Map(location=st.session_state['ubicacion'], zoom_start=12)
folium.Marker(st.session_state['ubicacion'], tooltip='Alojamiento').add_to(mapa)
clic = st_folium(mapa, height=450, use_container_width=True, returned_objects=['last_clicked'])

# Si el usuario hizo clic en un punto nuevo, se guarda y se vuelve a dibujar el mapa con el marcador
if clic and clic['last_clicked']:
    nueva = [clic['last_clicked']['lat'], clic['last_clicked']['lng']]
    if nueva != st.session_state['ubicacion']:
        st.session_state['ubicacion'] = nueva
        st.rerun()

latitud, longitud = st.session_state['ubicacion']
st.write(f'Ubicación seleccionada: latitud {latitud:.4f}, longitud {longitud:.4f}')

fuera_de_nyc = not (40.49 <= latitud <= 40.92 and -74.26 <= longitud <= -73.70)
if fuera_de_nyc:
    st.warning('La ubicación está fuera de la ciudad de Nueva York. Selecciona un punto dentro de la ciudad.')

distrito = st.selectbox('Distrito', ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island'])

# ---------------- Alojamiento y anfitrión
st.header('Alojamiento y anfitrión')
tipo = st.selectbox('Tipo de alojamiento', ['Entire home/apt', 'Private room', 'Shared room'])
noches_minimas = st.number_input('Noches mínimas', min_value=1, max_value=365, value=2)
disponibilidad = st.number_input('Días disponibles al año', min_value=0, max_value=365, value=120)
num_resenas = st.number_input('Número de reseñas', min_value=0, value=10)
resenas_mes = st.number_input('Reseñas por mes', min_value=0.0, value=0.5)
num_anuncios = st.number_input('Número de anuncios del anfitrión', min_value=1, value=1)

if st.button('Estimar precio', disabled=fuera_de_nyc):
    datos = pd.DataFrame({
        'neighbourhood_group': [distrito],
        'latitude': [latitud],
        'longitude': [longitud],
        'room_type': [tipo],
        'minimum_nights': [noches_minimas],
        'number_of_reviews': [num_resenas],
        'reviews_per_month': [resenas_mes],
        'calculated_host_listings_count': [num_anuncios],
        'availability_365': [disponibilidad],
    })
    precio = modelo.predict(datos)[0]
    st.success(f'Precio estimado: USD {precio:.0f} por noche')
    st.write('El error típico del modelo es de unos 23 USD; úsalo como precio de referencia.')

# 5. Archivos para GitHub y Streamlit Community Cloud

Streamlit Community Cloud instala las librerías indicadas en `requirements.txt`. Las versiones de **scikit-learn** y **XGBoost** se fijan iguales a las de este entorno, que son las mismas con las que se entrenó el modelo; con versiones distintas, el archivo `.pkl` podría no cargarse. Las demás librerías se dejan sin versión fija para evitar errores de instalación.

**Importante:** al crear la app en Streamlit Community Cloud, en *Advanced settings* se debe elegir la **misma versión de Python** que muestra la celda siguiente. Si se usa otra (por ejemplo, una más nueva), las versiones fijadas de scikit-learn y XGBoost pueden no instalarse y la app falla con `ModuleNotFoundError`.

In [ ]:
import sys
import sklearn
import xgboost

print('Python:', sys.version)
print('scikit-learn:', sklearn.__version__)
print('xgboost:', xgboost.__version__)

with open('requirements.txt', 'w') as f:
    f.write('streamlit\n')
    f.write('streamlit-folium\n')
    f.write('folium\n')
    f.write('joblib\n')
    f.write('pandas\n')
    f.write('numpy\n')
    f.write('scikit-learn==' + sklearn.__version__ + '\n')
    f.write('xgboost==' + xgboost.__version__ + '\n')

print()
print(open('requirements.txt').read())

Se descargan los 4 archivos que deben subirse al repositorio de GitHub, en la misma carpeta:

* `app.py` (archivo principal de la aplicación)
* `features.py`
* `modelo_final_airbnb.pkl`
* `requirements.txt`

In [ ]:
from google.colab import files
files.download('app.py')
files.download('features.py')
files.download('modelo_final_airbnb.pkl')
files.download('requirements.txt')